## MLFLOW & Modélisation

### Import des modules

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")
import mlflow
import mlflow.sklearn 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, roc_curve, precision_recall_curve, classification_report, confusion_matrix)



In [ ]:
# Importation des données 
app_train_clean = joblib.load("data/app_train_clean_v2.joblib")
app_test_clean = joblib.load("data/app_test_clean_v2.joblib")

In [ ]:
# Fonction permettant de vérifier la différence de colonnes entre 2 tables
def check_columns(df_ref, df_test, name_ref="train", name_test="test"):
    cols_ref = set(df_ref.columns)
    cols_test = set(df_test.columns)

    missing = cols_ref - cols_test
    extra = cols_test - cols_ref

    print(f"--- Vérification colonnes : {name_test} vs {name_ref} ---")
    print(f"Colonnes attendues : {len(cols_ref)}")
    print(f"Colonnes trouvées  : {len(cols_test)}")

    if missing:
        print("\n❌ Colonnes manquantes dans", name_test, ":")
        for c in sorted(missing):
            print("   -", c)
    else:
        print("\n✔️ Aucune colonne manquante")

    if extra:
        print("\n⚠️ Colonnes supplémentaires dans", name_test, ":")
        for c in sorted(extra):
            print("   -", c)
    else:
        print("\n✔️ Aucune colonne supplémentaire")

    print("\n--------------------------------------------\n")


In [5]:
# Check des colonnes entre train et test
check_columns(app_train_clean, app_test_clean, name_ref="train", name_test="test")

--- Vérification colonnes : test vs train ---
Colonnes attendues : 328
Colonnes trouvées  : 327

❌ Colonnes manquantes dans test :
   - TARGET

✔️ Aucune colonne supplémentaire

--------------------------------------------



Mon objectif principal est de pouvoir tracer l'impact du preprocessing, des modèles et des hyperparamètres sur les métriques et le scoring.

De ce fait il faudrait une structure du MLFlow permettant de:
- comparer facilement les préprocessings entre eux
- comparer facilement les modèles entre eux
- comparer facilement les hyperparamètres entre eux
- garder une hiérarchie lisible dans MLflow UI
- ne pas exploser le nombre de runs

### Fonctions du preprocessing

3 fonctions de preprocessing permettant de gérer les variables manquantes:
- Garder les NaN tel quel en les flaguant (je ne sais pas si en flaguant les numériques par '-1' çà n'a pas un impact sur les données?)
- Imputation classique(**médiane** pour les numériques, **mode** pour les catégorielles)
- Suppression des variables dont le ratio NaN est >= 70%

1 fonction pour tester l'impact de la gestion des outliers sur les scores et résultats en plus des nettoyages précédents

In [6]:
# Flag "Missing"
def preprocess_missing_flag(df):
    df = df.copy()
    df_num = df.select_dtypes(include=["int64", "float64"])
    df_cat = df.select_dtypes(include=["object", "category", "bool"])

    df_num = df_num.fillna(-1)
    df_cat = df_cat.fillna("MISSING")

    return pd.concat([df_num, df_cat], axis=1)


# Imputation classique
from sklearn.impute import SimpleImputer

def preprocess_imputation(df):
    df = df.copy()
    df_num = df.select_dtypes(include=["int64", "float64"])
    df_cat = df.select_dtypes(include=["object", "category", "bool"])

    df_num = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(df_num),
                          columns=df_num.columns)
    df_cat = pd.DataFrame(SimpleImputer(strategy="most_frequent").fit_transform(df_cat),
                          columns=df_cat.columns)

    return pd.concat([df_num, df_cat], axis=1)


# Suppression colonnes >= 70% NaN
def preprocess_drop70(df, threshold=0.7):
    df = df.copy()
    missing_ratio = df.isna().mean()
    cols_to_drop = missing_ratio[missing_ratio >= threshold].index.tolist()

    print("Colonnes supprimées (>=70% NaN) :", cols_to_drop)

    df = df.drop(columns=cols_to_drop)

    # puis imputation classique
    return preprocess_imputation(df)


# Gestion des outliers
def clip_outliers(df, lower_q=0.01, upper_q=0.99):
    df_out = df.copy()
    num_cols = df_out.select_dtypes(include=['int64', 'float64']).columns
    
    for col in num_cols:
        lower = df_out[col].quantile(lower_q)
        upper = df_out[col].quantile(upper_q)
        df_out[col] = df_out[col].clip(lower, upper)
    
    return df_out


def preprocess_outliers_missing_flag(X):
    X2 = clip_outliers(X)
    return preprocess_missing_flag(X2)

def preprocess_outliers_imputation(X):
    X2 = clip_outliers(X)
    return preprocess_imputation(X2)

def preprocess_outliers_drop70(X):
    X2 = clip_outliers(X)
    return preprocess_drop70(X2)


### Transformation des colonnes selon leur type: ColumnTransformer

Après la séparation des données en X & y:
- je crèe des listes de colonnes selon leur type (**numériques, catégorielle, booléennes**)
- ensuite j'applique la stratégie de transformation selon le type de la variable
    - les variables catégorielles ont 2 stratégies distinctes dû au nombre élevé de catégories pour certaines variables(
        **OneHotEncoder** si un *maximum de 10 catégories*
        **TargetEncoder** si *plus de 10 catégories*)

In [7]:
# Séparation du jeu de données
X = app_train_clean.drop(columns=["TARGET"])
y = app_train_clean["TARGET"].astype(int)

In [ ]:
# Fonction qui construit le process de transformation des colonnes
def tranformation_process(X_prep):
    num_cols = X_prep.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat_cols = X_prep.select_dtypes(include=["object", "category"]).columns.tolist()
    bool_cols = X_prep.select_dtypes(include=["bool"]).columns.tolist()

    nb_faible_cols = [c for c in cat_cols if X_prep[c].nunique() <= 10]
    nb_eleve_cols = [c for c in cat_cols if X_prep[c].nunique() > 10]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("card_faible", OneHotEncoder(handle_unknown="ignore"), nb_faible_cols),
            ("card_eleve", TargetEncoder(), nb_eleve_cols),
            ("bool", "passthrough", bool_cols)
        ]
    )
    return preprocessor


### Score métier et optimisation

L'objectif métier de la modélisation est de créer un **scoring métier - score qui permet de minimiser le coût d'erreur de prédiction des FN & FP**
- FN = faux négatif = mauvais client prédit bon <> coût très élevé (perte financière)
- FP = faux positif = bon client prédit mauvais <> coût plus faible (manque à gagner)

Pour se faire, il faudra trouver le *seuil optimal* permettant de minimiser ce coût, car le seuil par défaut étant de 0.5 ne permet pas de capter toutes les erreurs. 
Ainsi, j'ai créé une fonction qui:
- teste plusieurs seuils; et pour chaque seuil:
    - elle convertit les probabilités en classes
    - calcule FN, FP
    - calcule le coût métier
    - garde le seuil qui minimise ledit coût

In [9]:
# Coût des classes
cout_FN = 10.0
cout_FP = 1.0

# Fonction du calcul du coût métier
def cout_metier(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel() # transformer la matrice en un tableau 1D
    cout_total = FN * cout_FN + FP * cout_FP
    return cout_total, cm

# Fonction d'optimisation du seuil
def best_seuil(y_true, y_proba):
    seuils = np.linspace(0.01, 0.99, 99) # 99 seuils allant de 0.01 à 0.99 avec un espace de 0.01 (0.01, 0.02, 0.03, ..., 0.99)
    best_s = 0.5
    best_cost = np.inf
    best_cm = None

    for s in seuils:
        y_pred = (y_proba >= s).astype(int)
        cost, cm = cout_metier(y_true, y_pred)
        if cost < best_cost:
            best_cost = cost
            best_s = s
            best_cm = cm

    return best_s, best_cost, best_cm


### Déclaration des modèles à comparer - Optimisation des hyperparamètres

In [10]:
# Grilles d'hyperparamètres pour optimisation

# LR
param_LR = {
    "model__C": [0.01, 0.1, 1, 10], 
    "model__max_iter": [200, 500]}

# Random Forest
param_RF = {
    # "model__n_estimators": [200, 400, 600],
    # "model__max_depth": [5, 10, 15, None],
    # "model__min_samples_split": [2, 5, 10],
    # "model__min_samples_leaf": [1, 2, 4]
    "model__max_depth": [3, 5],
    "model__n_estimators": [100, 150, 200],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]}

# XGBoost
param_XGB = {
    # "model__n_estimators": [400, 600, 700],
    # "model__learning_rate": [0.1, 0.2, 0.3],
    # "model__max_depth": [3, 5, 7],
    # "model__subsample": [0.8, 1.0],
    # "model__colsample_bytree": [0.8, 1.0],
    # "model__min_child_weight": [1, 3, 5],
    # "model__gamma": [0, 1, 5],
    # "model__reg_lambda": [1, 5, 10]
    "model__learning_rate": [0.05, 0.1], 
    "model__max_depth": [3, 5], 
    "model__subsample": [0.8, 1.0], 
    "model__colsample_bytree": [0.8, 1.0]}

In [11]:
# Rapport entre négatifs et positifs - pour calculer le ratio des classes négatives et positives afin de l'utiliser dans l'hyperparamètre
scale_pos_weight = y.value_counts()[0] / y.value_counts()[1]

# Modèles à comparer + hyperparamètres
modeles = {
    "Dummy": (DummyClassifier(strategy = "stratified"), {}),
    "LogisticRegression": (LogisticRegression(max_iter = 2000, class_weight = "balanced", C = 0.1), param_LR),

    "RandomForest": (RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state = 42,max_depth = 3, min_samples_split = 2, min_samples_leaf = 2),
                    param_RF),

    "XGBoost": (XGBClassifier(eval_metric="logloss", n_estimators = 100, learning_rate = 0.01, max_depth = 3, subsample = 0.4, colsample_bytree = 0.4, min_child_weight = 3, gamma = 2, 
                             reg_lambda = 3, scale_pos_weight = scale_pos_weight, random_state = 42),
                param_XGB)}


In [ ]:

def optimisation_hyperparametres(model_name, model, param_grid, preprocessor, X_train, y_train):

    # 1. Réduire n_estimators uniquement pour l’optimisation
    if hasattr(model, "n_estimators"):
        # Valeurs recommandées pour accélérer
        if model_name.lower().startswith("randomforest"):
            model.set_params(n_estimators=150)
        elif model_name.lower().startswith("xgb"):
            model.set_params(n_estimators=200)

    # 2. Pipeline
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    # 3. RandomizedSearchCV plus léger
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_grid,
        n_iter=5,               
        scoring="roc_auc",
        cv=3,                 
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    search.fit(X_train, y_train)

    return search.best_estimator_, search.best_params_, search.best_score_




#### Fonction d'évaluation des modèles - Pipeline MLOps

Fonction qui pour chaque modèle:
- le combine avec le préprocesseur intégré (pipeline preprocessing), 
- l’entraîne, 
- calcule toutes les métriques utiles, 
- optimise le seuil métier, 
- génère les artefacts (ROC + matrice de confusion),
- enregistre tout dans MLflow.

Ceci est effectué en mettant en place un tracking permettant d'enregistrer, pour chaque modèle, les résultats obtenus dans le but de les comparer selon le besoin:
- démarrer un **run MLFlow** en créant un sous-run par modèle *(with mlflow.start_run(run_name=model_name, nested=True))* afin d'enregistrer tout ce qui lui est demandé
    - enregistrer les hyperparamètres
    - entrainer le pipeline <> application du preprocessing dans le pipeline
    - calculer les prédictions (0, 1) & probabilités (entre 0 & 1)
    - calculer les métriques nécessaires, puis les enregister dans le MLFlow <> j'ai choisi de mesurer la qualité globale, la capacité à détecter les mauvais clients (*Recall*), l'équilibre précision/rappel (*F1*), et 
    surtout la performance globale (*AUC - la contrainte est de s'assurer qu'il soit inférieur à 0.82, indicateur de surapprentissage*)
    - effectuer une validation croisée indépendamment du split train/test
    - calculer, pour 99 seuils distinct, le coût métier, puis choisir le seuil qui minimise le coût. Le modèle qui permettra d'obtenir le coût total le moins élevé sera celui choisi
    - puis tracer la matrice de confusion et la courbe ROC.


In [ ]:
# Fonction permettant de calculer les métriques et artefacts(matrice de confusion, ROC), coût métier, seuil optimal
def evaluation_modele(model_name, model, X_train, X_test, y_train, y_test, preprocessor):
    
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)])

    # Log hyperparamètres
    for param_name, param_value in model.get_params().items():
        mlflow.log_param(param_name, param_value)

    # Fit
    pipe.fit(X_train, y_train)

    # Prédictions
    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)

    y_proba_train = pipe.predict_proba(X_train)[:, 1]
    y_proba_test = pipe.predict_proba(X_test)[:, 1]

    # compteur de step pour éviter les collisions MLflow 
    step = 0

    # Métriques train/test
    metrics = {
            "accuracy_train": accuracy_score(y_train, y_pred_train),
            "accuracy_test": accuracy_score(y_test, y_pred_test),
            "precision_train": precision_score(y_train, y_pred_train),
            "precision_test": precision_score(y_test, y_pred_test),
            "recall_train": recall_score(y_train, y_pred_train),
            "recall_test": recall_score(y_test, y_pred_test),
            "f1_train": f1_score(y_train, y_pred_train),
            "f1_test": f1_score(y_test, y_pred_test),
            "auc_train": roc_auc_score(y_train, y_proba_train),
            "auc_test": roc_auc_score(y_test, y_proba_test)}

    for m, v in metrics.items():
        mlflow.log_metric(m, v, step=step)

    # Validation croisée
    cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
    auc_cv = cross_val_score(pipe, X_train, y_train, cv = cv, scoring = "roc_auc", n_jobs = -1 ).mean()   
    mlflow.log_metric("auc_cv", auc_cv, step=step)
    step += 1
    

    # Score métier
    best_s, best_cost, best_cm = best_seuil(y_test, y_proba_test)
    mlflow.log_metric("business_cost", best_cost, step=step); step += 1
    mlflow.log_metric("best_threshold", best_s, step=step); step += 1

    # Prédictions au seuil optimal
    y_pred_opt = (y_proba_test >= best_s).astype(int)

    # Recall des mauvais payeurs au seuil optimal
    recall_bad_payers = recall_score(y_test, y_pred_opt, pos_label=1)
    mlflow.log_metric("recall_bad_payers", recall_bad_payers, step=step)
    step += 1


    # Matrice de confusion
    fig_cm, ax = plt.subplots(figsize=(4,4))
    sns.heatmap(best_cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    plt.tight_layout()
    cm_path = f"artefacts/cm_{model_name}.png"
    fig_cm.savefig(cm_path)
    mlflow.log_artifact(cm_path)
    plt.close(fig_cm)


    # Courbe ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba_test)
    fig_roc, ax = plt.subplots(figsize=(5,4))
    ax.plot(fpr, tpr, label=f"AUC={metrics['auc_test']:.3f}")
    ax.plot([0,1],[0,1],"k--")
    plt.tight_layout()
    roc_path = f"artefacts/roc_{model_name}.png"
    fig_roc.savefig(roc_path)
    mlflow.log_artifact(roc_path)
    plt.close(fig_roc)

    # Log du modèle
    mlflow.sklearn.log_model(pipe, model_name)

    return metrics, auc_cv, best_s, best_cost


#### Exécution du MLFlow

In [ ]:
# Forcer MLflow à utiliser le dossier mlruns pour sauvegarder les runs sans passer par mlflow db
mlflow.set_tracking_uri("file:///C:/Users/Lenovo/Documents/WCS/GitHub/Classification-Risque-Credit-Pipeline-MLOps-/mlruns")


preprocessings = {
    "missing_flag": preprocess_missing_flag,
    "imputation": preprocess_imputation,
    "drop70": preprocess_drop70,
    "outliers_missing_flag": preprocess_outliers_missing_flag, 
    "outliers_imputation": preprocess_outliers_imputation, 
    "outliers_drop70": preprocess_outliers_drop70}

version = "v5_2-07-03"

for prep_name, prep_fn in preprocessings.items():

    # Sans optimisation
    mlflow.set_experiment(f"{prep_name}_base_{version}")

    with mlflow.start_run(run_name = f"{prep_name}_base_{version}"):

        X_prep = prep_fn(X)

        X_train, X_test, y_train, y_test = train_test_split(X_prep, y, test_size = 0.2, stratify = y, random_state = 4)

        # Sauvegarde pour SHAP
        X_test.to_csv(f"data/X_test_prep_{prep_name}.csv", index=False)
        y_test.to_csv(f"data/y_test_prep_{prep_name}.csv", index=False)

        preprocessor = tranformation_process(X_prep)
        

        # Log du dataset sans optimisation
        mlflow.log_input(mlflow.data.from_pandas(X_train, source=f"{prep_name}_train_{version}"))

        for model_name, (model, _) in modeles.items():
            print(f"Modèle sans optimisation: {model_name}")

            with mlflow.start_run(run_name = f"{model_name}_{version}", nested = True):
                evaluation_modele(model_name=model_name, model=model, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, preprocessor=preprocessor)


    # Avec optimisation des hyperparamètres
    mlflow.set_experiment(f"{prep_name}_optimisé_{version}")

    with mlflow.start_run(run_name = f"{prep_name}_optimisé_{version}"):

        X_prep = prep_fn(X)

        X_train, X_test, y_train, y_test = train_test_split(X_prep, y, test_size = 0.2, stratify = y, random_state = 4)

        # Sauvegarde pour SHAP
        X_test.to_csv(f"data/X_test_prep_{prep_name}.csv", index=False)
        y_test.to_csv(f"data/y_test_prep_{prep_name}.csv", index=False)

        preprocessor = tranformation_process(X_prep)
  

        # Log du dataset 
        mlflow.log_input(mlflow.data.from_pandas(X_train, source=f"{prep_name}_train_{version}"))

        for model_name, (model, param_grid) in modeles.items():
            print(f"Optimisation du modèle : {model_name}")

            meilleur_modele, meilleur_params, meilleur_score = optimisation_hyperparametres( model_name, model, param_grid, preprocessor, X_train, y_train )

            with mlflow.start_run(run_name=f"{model_name}_optimized_{version}", nested=True):

                mlflow.log_params(meilleur_params) 
                mlflow.log_metric("auc_cv_meilleur", meilleur_score)

                evaluation_modele(model_name=f"{model_name}_optimized", model=meilleur_modele.named_steps["model"], X_train=X_train, X_test=X_test, 
                                  y_train=y_train, y_test=y_test, preprocessor=preprocessor)

                # Enregistrement dans le Model Registry 
                mlflow.sklearn.log_model( meilleur_modele, artifact_path="model", registered_model_name=f"{prep_name}_{model_name}_optimized_{version}" )



2026/03/07 23:38:53 INFO mlflow.tracking.fluent: Experiment with name 'missing_flag_base_v5_2-07-03' does not exist. Creating a new experiment.


Modèle sans optimisation: Dummy


2026/03/07 23:45:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Modèle sans optimisation: LogisticRegression


2026/03/07 23:49:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/03/08 07:09:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 07:09:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'outliers_drop70_LogisticRegression_optimized_v5_2-07-03'.
Created version '1' of model 'outliers_drop70_LogisticRegression_optimized_v5_2-07-03'.


Optimisation du modèle : RandomForest
Fitting 3 folds for each of 5 candidates, totalling 15 fits


2026/03/08 07:32:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 07:32:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'outliers_drop70_RandomForest_optimized_v5_2-07-03'.
Created version '1' of model 'outliers_drop70_RandomForest_optimized_v5_2-07-03'.


Optimisation du modèle : XGBoost
Fitting 3 folds for each of 5 candidates, totalling 15 fits


2026/03/08 07:46:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 07:46:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'outliers_drop70_XGBoost_optimized_v5_2-07-03'.
Created version '1' of model 'outliers_drop70_XGBoost_optimized_v5_2-07-03'.


In [ ]:
import os
os.listdir("C:/Users/Lenovo/Documents/WCS/GitHub/Classification-Risque-Credit-Pipeline-MLOps/mlruns")

In [15]:
mlflow.end_run()

In [67]:
# Extraction des données finales: scores, coût minimal, seuil optimal, recall de mauvais payeurs pour chaque test effectué et par modèle

def tous_les_runs():
    client = mlflow.tracking.MlflowClient()
    experiments = mlflow.search_experiments()   # ✔️ compatible toutes versions

    rows = []

    for exp in experiments:
        runs = client.search_runs(exp.experiment_id)

        for run in runs:
            data = run.data

            row = {
                "experiment": exp.name,
                "run_id": run.info.run_id,
                "run_name": run.info.run_name,
                "preprocessing": exp.name.split("_")[0],
                "mode": "optimisé" if "optim" in exp.name else "base",
                "model": run.info.run_name.replace("_optimized", ""),
                "auc": data.metrics.get("auc_test"),
                "seuil_optimal": data.metrics.get("best_threshold"),
                "cout_minimal": data.metrics.get("business_cost"),
                "précision": data.metrics.get("precision_test"),
                "recall_bad_payers": data.metrics.get("recall_bad_payers"),
                "params": data.params
            }

            rows.append(row)   # ✔️ tu avais oublié d'ajouter la ligne !

    return pd.DataFrame(rows)

df_results = tous_les_runs()
df_results["auc"] = df_results["auc"].round(2)
df_results_sorted = df_results.sort_values(by=["cout_minimal", "auc"], ascending=[True, False])
df_results_sorted



,experiment,run_id,run_name,preprocessing,mode,model,auc,seuil_optimal,cout_minimal,précision,recall_bad_payers,params
21,outliers_missing_flag_optimisé_v5_2-07-03,79a042a4457d43c5a89a121722be0a30,XGBoost_optimized_v5_2-07-03,outliers,optimisé,XGBoost_v5_2-07-03,0.78,0.50,30629.0,0.188498,0.672709,"{'base_score': 'None', 'booster': 'None', 'cal..."
112,missing_flag_optimisé_v5-07-03,cd71d3352dad456f88a07b1a963d3829,XGBoost_optimized_v5-07-03,missing,optimisé,XGBoost_v5-07-03,0.78,0.47,30653.0,0.187219,0.715408,"{'base_score': 'None', 'booster': 'None', 'cal..."
82,outliers_missing_flag_optimisé_v5-07-03,3cbe62ccc9344fbaae6cbfee303d3690,XGBoost_optimized_v5-07-03,outliers,optimisé,XGBoost_v5-07-03,0.78,0.47,30726.0,0.187479,0.710574,"{'base_score': 'None', 'booster': 'None', 'cal..."
51,missing_flag_optimisé_v5_2-07-03,bd607866a59f4ee6a2885ce926a59439,XGBoost_optimized_v5_2-07-03,missing,optimisé,XGBoost_v5_2-07-03,0.78,0.49,30771.0,0.186758,0.684189,"{'base_score': 'None', 'booster': 'None', 'cal..."
72,outliers_imputation_optimisé_v5-07-03,fe9a86b14f514878be5df6bf2692e663,XGBoost_optimized_v5-07-03,outliers,optimisé,XGBoost_v5-07-03,0.78,0.51,30771.0,0.187865,0.657200,"{'base_score': 'None', 'booster': 'None', 'cal..."
...,...,...,...,...,...,...,...,...,...,...,...,...
133,imputation_optimisé_v4-07-03,98555b84ce5244eb9543e5abd4ff61e6,imputation_optimisé_v4-07-03,imputation,optimisé,imputation_optimisé_v4-07-03,NaN,NaN,NaN,NaN,NaN,{}
138,imputation_base_v4-07-03,3d975f8448a544fd945a2820eaf6b6a4,imputation_base_v4-07-03,imputation,base,imputation_base_v4-07-03,NaN,NaN,NaN,NaN,NaN,{}
143,missing_flag_optimisé_v4-07-03,1c85cdcc54d94edba7e757ebab6520b2,missing_flag_optimisé_v4-07-03,missing,optimisé,missing_flag_optimisé_v4-07-03,NaN,NaN,NaN,NaN,NaN,{}
144,missing_flag_base_v4-07-03,09350d3ffa5a490c82cd576116cf480a,missing_flag_base_v4-07-03,missing,base,missing_flag_base_v4-07-03,NaN,NaN,NaN,NaN,NaN,{}


### Meilleur modèle

Après comparaison des modèles entre les éxpériences et les runs exécutés, le modèle **XGBoost** est celui qui a obtenu, toutes expériences confondues:
- le meilleur coût minimal ***30629 €***
- au seuil optimal de ***0,5***
- avec un scor AUC de ***78%***
- un taux de recall de ***67,3%***

In [58]:
def clean_experiment_name(exp_name):
    # Supprime la partie version à la fin (ex: _v5_2-07-03)
    parts = exp_name.split("_v")
    return parts[0]  # garde tout avant _v...

df_results_sorted["nom_modele_ppt"] = df_results_sorted.apply(lambda row: f"{clean_experiment_name(row['experiment'])}_{row['model']}", axis=1)


In [59]:
df_v6 = df_results_sorted[df_results_sorted['experiment'].str.contains('v5_2', na=False)]
len(df_v6)

61

In [51]:
df_results_sorted.to_csv("data/resultats_runs_v4.csv")

In [18]:
# Enregistrer le meilleur modèle pour utilisation ultérieure + pipeline transformation de données
joblib.dump(meilleur_modele, "best_model.joblib")
# joblib.dump(preprocessor, "preprocessor.joblib")
mlflow.log_artifact("best_model.joblib")


#### Prédictions finales + décision de la demande

In [19]:
expected_cols = preprocessor.feature_names_in_
missing = set(expected_cols) - set(app_train_clean.columns)
missing

set()

In [20]:
expected_cols = preprocessor.feature_names_in_
missing = set(expected_cols) - set(app_test_clean.columns)
missing

set()

In [ ]:
# Récupération du meilleur run
best_row = df_results_sorted.iloc[0]
best_run_id = best_row["run_id"]
best_threshold = best_row["seuil_optimal"]

# Sauvegarde du seuil et du run_id
joblib.dump(best_threshold, "best_threshold.joblib")

with open("best_run_id.txt", "w") as f:
    f.write(best_run_id)

with open("best_model_name.txt", "w") as f:
    f.write(best_row["model"])   # garder le nom du modèle


# Chargement du pipeline complet depuis MLflow - charge le préprocesseur + le modèle déjà fit
pipe = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")


# Prédictions finales sur app_test_clean
y_pred_final = pipe.predict_proba(app_test_clean)[:, 1]

# Application du seuil optimal
decision = (y_pred_final >= best_threshold).astype(int)


# Construction du dataframe final
pred_finales3 = pd.DataFrame({
    "SK_ID_CURR": app_test_clean["SK_ID_CURR"],
    "TARGET": y_pred_final,
    "DECISION": decision
})

pred_finales3["DECISION"] = pred_finales3["DECISION"].map({
    1: "ACCORDÉ",
    0: "REFUSÉ"
})

pred_finales3.head(15)


,SK_ID_CURR,TARGET,DECISION
0,100001,0.144826,REFUSÉ
1,100005,0.523025,ACCORDÉ
2,100013,0.084557,REFUSÉ
3,100028,0.239340,REFUSÉ
4,100038,0.266300,REFUSÉ
5,100042,0.099464,REFUSÉ
6,100057,0.058770,REFUSÉ
7,100065,0.106076,REFUSÉ
8,100066,0.172418,REFUSÉ
9,100067,0.567248,ACCORDÉ


In [22]:
pred_finales3.to_csv("data/pred_finales3.csv")

In [23]:
# vérifier que le run existe dans MLFlow
client = mlflow.tracking.MlflowClient()
run = client.get_run(best_run_id)
print(run.info.run_id, run.info.run_name)

# Vérifier que le modèle chargé correspond au run
print(meilleur_modele)

# Vérifier le seuil optimal
print(best_threshold)


79a042a4457d43c5a89a121722be0a30 XGBoost_optimized_v5_2-07-03
Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['SK_ID_CURR',
                                                   'BUREAU_SK_ID_BUREAU_max',
                                                   'POS_SK_DPD_min',
                                                   'PREV_NAME_TYPE_SUITE_Family',
                                                   'PREV_DAYS_FIRST_DRAWING_sum',
                                                   'PREV_NAME_GOODS_CATEGORY_Other',
                                                   'DAYS_LAST_PHONE_CHANGE',
                                                   'PREV_DAYS_FIRST_DRAWING_max',
                                                   'PREV_PRODUCT_COMBINATION_POS '
                                                   'others without interest',
                                                   